# Russell (Nicaragua) α-β — Preprocessing

Build one cloud per (donor, chain) from Russell's TCRdist-format clonotype files.

**Key differences from Rosati:**
- TCRdist format (not MiXCR): columns `cdr3`, `v_gene`, `j_gene`, `productive` (no readCount/UMI columns).
- Already filtered to **UMI≥5** (see `umi5` in filenames); the count column was dropped after filtering.
- **No abundance information** → **uniform weights** (w = 1/N). The descriptor cannot weight by clonal
  expansion because the data does not preserve it. Clouds remain valid repertoires; they just lose the
  abundance dimension. Single version (no UMI sweep).
- 150 donors, all with both chains; depth ~4k-26k clonotypes.

## 1 — Paths + cleaning functions

In [ ]:
import os, re, glob
import numpy as np
import pandas as pd

RA = '/home/immunologylab/bioinformatics/raw_data/russell_hla/nicaragua_data/TCRA'
RB = '/home/immunologylab/bioinformatics/raw_data/russell_hla/nicaragua_data/TCRB'
OUTDIR = '/home/immunologylab/bioinformatics/analysis/data/processed/russell/clouds_raw'
os.makedirs(OUTDIR, exist_ok=True)

MIN_CDR3 = 8

def clean_vgene(v):
    """'TRAV1-1*01' -> 'TRAV1-1'.  Handles 'TRAV38-2/DV8*01' -> 'TRAV38-2/DV8'."""
    if not isinstance(v, str) or not v:
        return None
    return v.split('*')[0]

def preprocess_one(path, expected_locus):
    """TCRdist format -> productive filter -> locus filter -> dedup -> uniform weights."""
    df = pd.read_csv(path, sep='\t', low_memory=False)
    # productive == True
    df = df[df['productive'].astype(str).str.lower() == 'true']
    # CDR3 aa: starts with C, length >= 8, no stop/frameshift chars
    df = df[df['cdr3'].notna()]
    df = df[df['cdr3'].str.startswith('C')]
    df = df[df['cdr3'].str.len() >= MIN_CDR3]
    df = df[~df['cdr3'].str.contains(r'[\*_]', regex=True)]
    # clean V gene
    df['v_clean'] = df['v_gene'].map(clean_vgene)
    df = df[df['v_clean'].notna()]
    # LOCUS filter: keep only expected chain (drop TRD/TRG leakage)
    df = df[df['v_clean'].str.startswith(expected_locus)]
    # dedup by (cdr3, v_clean) -- no counts, so just unique clonotypes
    g = df.drop_duplicates(subset=['cdr3', 'v_clean'])[['cdr3', 'v_clean']].copy()
    g.columns = ['cdr3aa', 'v_gene']
    if len(g) == 0:
        return None
    # UNIFORM weights (no abundance info available)
    g['count'] = 1
    g['w_log'] = 1.0 / len(g)
    return g[['cdr3aa', 'v_gene', 'count', 'w_log']]

print('functions ready')

## 2 — Test on one donor (verify locus filter + uniform weights)

In [ ]:
test_a = sorted(glob.glob(f'{RA}/*_A.tsv'))[0]
test_b = test_a.replace('/TCRA/', '/TCRB/').replace('_A.tsv', '_B.tsv')

for path, locus, name in [(test_a, 'TRA', 'alpha'), (test_b, 'TRB', 'beta')]:
    raw = pd.read_csv(path, sep='\t', low_memory=False)
    cloud = preprocess_one(path, locus)
    vraw = raw['v_gene'].map(clean_vgene).dropna()
    other = vraw[~vraw.str.startswith(locus)].str[:3].value_counts().to_dict()
    print(f'{name}: raw {len(raw)} -> clean {len(cloud)} clonotypes | non-{locus} loci: {other}')
    print(cloud.head(3).to_string())
    print()

## 3 — Run all 150 donors × 2 chains

In [ ]:
def donor_of(path):
    # 'nica_10270_run4_umi5_A.tsv' -> 'nica_10270'
    base = os.path.basename(path)
    return '_'.join(base.split('_')[:2])

rows, skipped = [], []
for path in sorted(glob.glob(f'{RA}/*_A.tsv')):
    donor = donor_of(path)
    pb = path.replace('/TCRA/', '/TCRB/').replace('_A.tsv', '_B.tsv')
    for p, locus, ch in [(path, 'TRA', 'A'), (pb, 'TRB', 'B')]:
        if not os.path.exists(p):
            skipped.append((donor, ch, 'no file')); continue
        cloud = preprocess_one(p, locus)
        if cloud is None or len(cloud) < 10:
            skipped.append((donor, ch, 'too few')); continue
        stem = f'{donor}_{ch}'
        cloud.to_parquet(f'{OUTDIR}/{stem}.parquet', index=False)
        rows.append({'stem': stem, 'donor': donor, 'chain': locus, 'n_clonotypes': len(cloud)})

summary = pd.DataFrame(rows)
summary.to_csv(f'{OUTDIR}/../clouds_summary.tsv', sep='\t', index=False)
print('clouds written:', len(summary))
print('skipped:', len(skipped), skipped[:10])

## 4 — Summary: complete pairs and depth α vs β

In [ ]:
wide = summary.pivot_table(index='donor', columns='chain', values='n_clonotypes')
pairs = wide.dropna()
print(f'complete pairs (both chains): {len(pairs)} of {summary["donor"].nunique()} donors')
print('\ndepth (n_clonotypes) by chain:')
print(summary.groupby('chain')['n_clonotypes'].describe()[['count','min','25%','50%','75%','max']])
wide['ratio_A_B'] = wide['TRA'] / wide['TRB']
print(f"\nmedian TRA/TRB depth ratio: {wide['ratio_A_B'].median():.2f}")

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8.5, 5))
ax.hist(summary[summary['chain']=='TRA']['n_clonotypes'], bins=30, alpha=0.6, color='#2C7FB8', label='TRA (α)', edgecolor='white')
ax.hist(summary[summary['chain']=='TRB']['n_clonotypes'], bins=30, alpha=0.6, color='#C0392B', label='TRB (β)', edgecolor='white')
ax.set_xlabel('Clonotypes per cloud', fontsize=12, fontweight='bold')
ax.set_ylabel('Number of clouds', fontsize=12, fontweight='bold')
ax.set_title('Russell cloud size distribution (UMI≥5, uniform weights)', fontsize=12)
ax.legend(fontsize=10); ax.grid(True, alpha=0.25)
plt.tight_layout(); plt.show()